## Chatbot com DialoGPT

### 📌 Objetivo:
Este notebook demonstra como utilizar o modelo pré-treinado DialoGPT da Microsoft, disponível na Hugging Face, para construir um chatbot simples de perguntas e respostas.

O modelo utilizado é o `microsoft/DialoGPT-medium`, que foi treinado em diálogos de fóruns como o Reddit. A abordagem consiste em usar o pipeline da Hugging Face para processar interações simples de forma rápida, com poucas linhas de código.


### Importação de bibliotecas

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import pipeline

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

### Carregamento do modelo e tokenizador
Vamos usar um modelo treinado para conversação:

In [10]:
modelo_nome = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(modelo_nome)
modelo = AutoModelForCausalLM.from_pretrained(modelo_nome)

### Função de conversação

In [11]:
def responder_ao_usuario(pergunta, chat_historia_ids=None):
    nova_entrada_ids = tokenizer.encode(pergunta + tokenizer.eos_token, return_tensors="pt")
    input_ids = torch.cat([chat_historia_ids, nova_entrada_ids], dim=-1) if chat_historia_ids is not None else nova_entrada_ids

    resposta_ids = modelo.generate(
        input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95
    )

    resposta = tokenizer.decode(resposta_ids[:, input_ids.shape[-1]:][0], skip_special_tokens=True)
    return resposta, resposta_ids

### 💬 Teste de frase

In [12]:
frase_teste = "I am not happy with the product at all."
detector = pipeline("sentiment-analysis")
resultado = detector(frase_teste)[0]
print("Frase:", frase_teste)
print(f"Sentimento: {resultado['label']} ({resultado['score']:.2%})")

Frase: I am not happy with the product at all.
Sentimento: NEGATIVE (99.98%)


### Loop de interação

In [5]:
chat_historia_ids = None

print("Chatbot: Olá! Como posso ajudar? (Digite 'sair' para encerrar)")
while True:
    entrada = input("Você: ")
    if entrada.lower() == "sair":
        break
    resposta, chat_historia_ids = responder_ao_usuario(entrada, chat_historia_ids)
    print(f"Chatbot: {resposta}")

Chatbot: Olá! Como posso ajudar? (Digite 'sair' para encerrar)


Você:  how can i use this chat ?


Chatbot: In game settings at the start of the day.


Você:  sair


## Conclusão

Este experimento demonstrou o uso do modelo `DialoGPT-medium` da Microsoft para criar um chatbot simples com o pipeline da Hugging Face. O código funcionou corretamente, permitindo que perguntas fossem enviadas e o modelo respondesse em tempo real.

No entanto, é importante destacar que:

- O modelo DialoGPT foi treinado em conversas genéricas (como do Reddit) e não possui objetivo de gerar respostas úteis ou informativas em contextos específicos.
- Algumas respostas podem parecer vagas, incoerentes ou fora de contexto. Isso não indica erro no código, mas sim uma limitação natural do modelo.
- Não há memória de conversação entre interações. Cada pergunta é tratada isoladamente.

### 🔁 Modelos Alternativos

Se o objetivo for construir um chatbot mais robusto, com maior capacidade de instrução e coerência, considere utilizar modelos mais modernos e ajustados para diálogo, como:

| Modelo | Hugging Face ID | Características |
|--------|------------------|-----------------|
| LLaMA 2 Chat | `meta-llama/Llama-2-7b-chat` | Treinado para conversação útil |
| Mistral Instruct | `mistralai/Mistral-7B-Instruct` | Leve, rápido e direto ao ponto |
| Gemma | `google/gemma-7b-it` | Instrução de alta qualidade com boa compatibilidade |

Esses modelos são maiores e podem exigir maior poder computacional, mas fornecem resultados mais úteis e naturais.

---

Este notebook serviu como ponto de partida para compreender o uso de modelos conversacionais com Hugging Face. Para aplicações reais, vale explorar opções de fine-tuning, memória de conversação e uso de modelos mais potentes.
